# Part I — Defining Distress and the Information Set

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Validate the supplied next-month target against first failure dates; identify the at-risk matched U.S. universe; diagnose censoring and company-month multiplicity; export the forecasting panel.

Run from the repository root (or its `notebooks/` subdirectory). See `README.md` for execution instructions. Source-data assumptions remain provisional pending TA confirmation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.6f}".format)

# Dataset location
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

file_path = project_root / 'Project_5_data_chs_main_global.parquet'

assert file_path.exists(), "Dataset not found."

print("Dataset located successfully.")

## Observation unit and source data

Use the CRSP security identifier `permno` and calendar `month_end` as the unique panel key. Retain `permco` and `gvkey` for company identification. The original run had no duplicate PERMNO-month keys, but multiple securities could represent one company. We keep security-month observations to avoid arbitrary deletion, report distinct PERMCO-month failures separately and recommend accounting for company-level dependence in model inference. This is a security-level implementation of the requested firm-month design.

The supplementary initial-training diagnostic found 2,494 excess security rows in 2,376 repeated PERMCO-month groups, with no conflicting observed binary outcomes. Selecting the largest contemporaneous market-equity security (tie-break: PERMNO) before filtering unknown labels retains 1,039,560 of 1,042,054 labeled observations and 2,205 of 2,207 positive outcomes. The small composition change supports retaining securities; it does not establish identical model performance. Report company-level event counts and account for company-level dependence in Part IV.

In [ ]:
id_cols = [
    "permno",
    "permco",
    "gvkey",
    "iid",
    "id"
]

date_cols = [
    "date",
    "month_end",
    "first_failure_month",
    "first_exit_month",
    "first_ambiguous_exit_month",
    "global_factor_eom",
    "global_factor_date"
]

target_cols = [
    "at_risk",
    "failure_current_month",
    "failure_next_month",
    "failure_t_plus_1",
    "failure"
]

sample_cols = [
    "global_factor_matched",
    "obs_main",
    "exch_main",
    "common",
    "primary_sec",
    "excntry",
    "crsp_shrcd",
    "crsp_exchcd",
    "source_crsp"
]

selected_cols = (
    id_cols + date_cols + target_cols + sample_cols
)

df = pd.read_parquet(
    file_path,
    columns=selected_cols
)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df = df.sort_values(
    ["permno", "month_end"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)

assert not df.duplicated(
    ["permno", "month_end"]
).any()

assert df["permno"].notna().all()
assert df["month_end"].notna().all()

print("Primary observation keys validated.")

assert df["month_end"].dt.is_month_end.all(), "Non-calendar month-end keys."
assert df["failure_t_plus_1"].dropna().isin([0, 1]).all()
assert df["at_risk"].isin([0, 1]).all()

## Failure definition and one-month horizon

Use the instructor-supplied `failure_t_plus_1` (renamed `target` on export), equal to one for a recorded first failure in the next calendar month and zero for a known non-failure. A one-month horizon is required by the assignment and corresponds to the CHS discrete-time hazard benchmark. CHS defines broad failure as bankruptcy, financially driven delisting or a D credit rating. The supplied panel does not independently identify those components, so we do not assert an exact match or a bankruptcy-only definition. Longer horizons are not part of this baseline.

In the supplementary diagnostic through November 1995, all 2,983 distinct recorded failure security-months coincide with recorded exits; no security appears after its recorded failure in the observed panel. This is consistent with the supplied terminal-event construction, but does not identify the event types. Pending TA confirmation, refer to the supplied corporate failure indicator rather than asserting the broad or narrow CHS definition.

In [ ]:
parquet_file = pq.ParquetFile(file_path)

all_columns = parquet_file.schema_arrow.names

keywords = [
    "fail",
    "bankrupt",
    "default",
    "rating",
    "exit",
    "delist",
    "dlst",
    "dlret",
    "censor",
    "risk"
]

event_columns = [
    col for col in all_columns
    if any(k in col.lower() for k in keywords)
]

print("Potential event-related columns:")

for col in event_columns:
    print(col)

Set `feature_month = month_end` and `forecast_month = feature_month + one calendar month`. Validate the supplied label against `first_failure_month` rather than shifting it again. This preserves the actual calendar horizon even when there is a gap in a security's history.

In [ ]:
df["feature_month"] = df["month_end"]

df["forecast_month"] = (
    df["feature_month"] + pd.offsets.MonthEnd(1)
)

display(
    df[
        [
            "permno",
            "feature_month",
            "forecast_month",
            "failure_current_month",
            "failure_t_plus_1"
        ]
    ].head(15)
)

In [ ]:
expected_failure = (
    df["first_failure_month"]
    .eq(df["forecast_month"])
    .astype("int8")
)

labeled = df["failure_t_plus_1"].notna()

alignment_table = pd.crosstab(
    expected_failure[labeled],
    df.loc[labeled, "failure_t_plus_1"],
    rownames=["Expected next-month failure"],
    colnames=["Supplied target"]
)

display(alignment_table)

mismatch = (
    labeled
    & df["failure_t_plus_1"].ne(expected_failure)
)

print("Target alignment mismatches:", mismatch.sum())

display(
    df.loc[
        mismatch,
        [
            "permno",
            "feature_month",
            "forecast_month",
            "first_failure_month",
            "failure_t_plus_1",
            "at_risk"
        ]
    ].head(20)
)

assert not mismatch.any(), "Supplied target and recorded first failure disagree."

## Risk set and unknown outcomes

Require `at_risk == 1` because the benchmark models failure conditional on being at risk at the feature date. Retain unknown outcomes as missing, not zero; otherwise censoring could bias probabilities downward. First failure, exit and ambiguous-exit dates are used only for diagnostics, never as predictors.

In [ ]:
risk_target_table = pd.crosstab(
    df["at_risk"],
    df["failure_t_plus_1"].fillna(-1)
)

risk_target_table = risk_target_table.rename(
    columns={
        -1.0: "Missing",
        0.0: "Non-failure",
        1.0: "Failure"
    }
)

display(risk_target_table)

print(
    "Positive targets outside risk set:",
    (
        df["at_risk"].eq(0)
        & df["failure_t_plus_1"].eq(1)
    ).sum()
)

print(
    "Labeled observations outside risk set:",
    (
        df["at_risk"].eq(0)
        & df["failure_t_plus_1"].notna()
    ).sum()
)

assert not (df["at_risk"].eq(0) & df["failure_t_plus_1"].notna()).any()

In [ ]:
missing_target = df.loc[
    df["failure_t_plus_1"].isna()
].copy()

missing_target_diagnostics = pd.Series({
    "Total missing targets": len(missing_target),

    "Outside risk set":
        missing_target["at_risk"].eq(0).sum(),

    "Inside risk set":
        missing_target["at_risk"].eq(1).sum(),

    "Feature month is December 2024":
        missing_target["feature_month"]
        .eq(pd.Timestamp("2024-12-31")).sum(),

    "Exit on or before forecast month":
        missing_target["first_exit_month"]
        .le(missing_target["forecast_month"]).sum(),

    "Ambiguous exit on or before forecast month":
        missing_target["first_ambiguous_exit_month"]
        .le(missing_target["forecast_month"]).sum(),

    "Failure on or before feature month":
        missing_target["first_failure_month"]
        .le(missing_target["feature_month"]).sum()
})

display(missing_target_diagnostics)

In [ ]:
company_panel = df.loc[
    df["permco"].notna(),
    [
        "permno",
        "permco",
        "month_end",
        "primary_sec",
        "global_factor_matched",
        "failure_t_plus_1"
    ]
]

duplicate_company = company_panel.loc[
    company_panel.duplicated(
        ["permco", "month_end"],
        keep=False
    )
]

print(
    "Rows belonging to repeated company-month groups:",
    len(duplicate_company)
)

print(
    "Positive targets in repeated company-month groups:",
    duplicate_company["failure_t_plus_1"].eq(1).sum()
)

matched_company = company_panel.loc[
    company_panel["global_factor_matched"].eq(1)
]

print(
    "Duplicate company-month keys in matched sample:",
    matched_company.duplicated(
        ["permco", "month_end"]
    ).sum()
)

positive_events = company_panel.loc[
    company_panel["failure_t_plus_1"].eq(1)
]

print(
    "Repeated positive company-month events:",
    positive_events.duplicated(
        ["permco", "month_end"]
    ).sum()
)

display(duplicate_company.head(20))

## Common U.S. universe and merge timing

Require the supplied match flag and U.S. common primary main observations on main exchanges (`excntry`, `common`, `primary_sec`, `obs_main`, `exch_main`). This defines a comparable universe for CHS and expanded models. Check that matched `global_factor_eom` equals `month_end` and neither source date exceeds that month-end. The original run matched 3,442,786 of 3,502,382 rows (98.30%). The supplementary check supports the candidate upstream key CHS (`permno`, `month_end`) to Global Characteristics (`id`, `eom`, retained as `global_factor_eom`): all 1,785,767 matched initial-training-window rows agree on numeric ID, calendar month-end and trading date; 1,771,231 comparable market equity pairs agree within tolerance. Both candidate security-month keys are unique among these matched rows. This is an inference from the merged file, not a reconstruction of the original join or proof of source-level uniqueness. Subsequent notebook joins explicitly use `permno` and `month_end`. Same-month alignment does not establish the publication timing of individual predictors.

In [ ]:
matched = df["global_factor_matched"].eq(1)

merge_diagnostics = pd.Series({
    "Total observations": len(df),

    "Matched observations": matched.sum(),

    "Unmatched observations": (~matched).sum(),

    "Missing Global Characteristics month-end":
        (
            matched
            & df["global_factor_eom"].isna()
        ).sum(),

    "Different Global Characteristics month-end":
        (
            matched
            & df["global_factor_eom"].notna()
            & df["global_factor_eom"].ne(df["month_end"])
        ).sum(),

    "Global Characteristics date after month-end":
        (
            matched
            & df["global_factor_date"].gt(df["month_end"])
        ).sum(),

    "CHS trading date after month-end":
        df["date"].gt(df["month_end"]).sum(),

    "Non-US or unknown country among matched":
        (
            matched
            & df["excntry"].ne("USA")
        ).sum(),

    "Non-primary or unknown security among matched":
        (
            matched
            & df["primary_sec"].ne(1)
        ).sum()
})

display(merge_diagnostics)

print(
    "Merge success rate:",
    f"{matched.mean():.4%}"
)

# A missing source date is distinct from a date that is too late.
merge_diagnostics.loc["Missing matched Global Characteristics source date"] = (
    matched & df["global_factor_date"].isna()
).sum()
merge_diagnostics.loc["Missing CHS trading date"] = df["date"].isna().sum()
# Enforce the date checks established by the original run.
assert merge_diagnostics["Missing Global Characteristics month-end"] == 0
assert merge_diagnostics["Different Global Characteristics month-end"] == 0
assert merge_diagnostics["Global Characteristics date after month-end"] == 0
assert merge_diagnostics["CHS trading date after month-end"] == 0
display(merge_diagnostics)

In [ ]:
sample_rules = {
    "Original dataset":
        pd.Series(True, index=df.index),

    "Matched Global Characteristics":
        df["global_factor_matched"].eq(1),

    "US common primary securities":
        (
            df["excntry"].eq("USA")
            & df["common"].eq(1)
            & df["primary_sec"].eq(1)
            & df["obs_main"].eq(1)
            & df["exch_main"].eq(1)
        ),

    "Population at risk":
        df["at_risk"].eq(1),

    "Known one-month-ahead outcome":
        df["failure_t_plus_1"].notna(),

    "Forecast month through December 2024":
        df["forecast_month"].le(
            pd.Timestamp("2024-12-31")
        )
}

eligible = pd.Series(True, index=df.index)

sample_records = []

for rule_name, condition in sample_rules.items():

    previous_count = eligible.sum()

    eligible &= condition.fillna(False)

    sample = df.loc[eligible]

    sample_records.append({
        "restriction": rule_name,
        "observations": len(sample),
        "removed": previous_count - len(sample),
        "unique_securities": sample["permno"].nunique(),
        "unique_companies": sample["permco"].nunique(),
        "known_outcomes":
            sample["failure_t_plus_1"].notna().sum(),
        "positive_observations":
            sample["failure_t_plus_1"].eq(1).sum(),
        "failure_rate_pct":
            sample["failure_t_plus_1"].mean() * 100
    })

sample_attrition = pd.DataFrame(sample_records)

display(sample_attrition)

## Exit and continuity diagnostics

Use a priority-based classification of unknown at-risk outcomes: beyond the sample end, ambiguous exit, other recorded exit, then other unknown. These diagnostic categories do not establish the underlying legal or financial cause of exit. No artificial months are inserted for entry, exit or gaps; mergers and nonfinancial versus financial delistings cannot be independently classified with the supplied event fields.

In [ ]:
# Examine unknown outcomes inside the risk set
unknown_at_risk = df.loc[
    df["at_risk"].eq(1)
    & df["failure_t_plus_1"].isna()
].copy()

sample_end = pd.Timestamp("2024-12-31")

conditions = [
    unknown_at_risk["forecast_month"].gt(sample_end),

    unknown_at_risk["first_ambiguous_exit_month"]
    .le(unknown_at_risk["forecast_month"]),

    unknown_at_risk["first_exit_month"]
    .le(unknown_at_risk["forecast_month"])
]

categories = [
    "End-of-sample right censoring",
    "Ambiguous exit",
    "Other recorded exit"
]

unknown_at_risk["censoring_category"] = np.select(
    conditions,
    categories,
    default="Other unknown outcome"
)

censoring_summary = (
    unknown_at_risk["censoring_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="observations")
)

display(censoring_summary)

In [ ]:
# Calendar month index
month_index = (
    df["feature_month"].dt.year * 12
    + df["feature_month"].dt.month
)

month_gap = month_index.groupby(
    df["permno"], sort=False
).diff()

continuity_summary = pd.Series({
    "First security observations":
        month_gap.isna().sum(),

    "Consecutive monthly observations":
        month_gap.eq(1).sum(),

    "Observations following a gap":
        month_gap.gt(1).sum(),

    "Securities with at least one gap":
        df.loc[month_gap.gt(1), "permno"].nunique(),

    "Non-positive month differences":
        month_gap.le(0).sum(),

    "Longest observed gap (months)":
        month_gap.max()
})

display(continuity_summary)

## Eligibility and export policy

Keep eligible at-risk observations whose forecast month is no later than December 2024. Separate forecast eligibility from supervised eligibility: the latter also requires a known binary target. The Part I output preserves unknown targets for possible forecasting; Part II adds feature-availability restrictions. This distinction avoids conflating censoring with survival.

## Security entry and exit coverage

Count first and last **observed** security-months in the raw supplied panel, separately from recorded event dates. Securities at the sample boundaries are not necessarily new listings or actual exits. We do not identify mergers or distinguish nonfinancial delistings without source event codes. These retrospective diagnostics do not enter the predictor matrix or change the risk set.

In [ ]:
life = df.groupby("permno", sort=False).agg(
    first_observed=("feature_month", "min"),
    last_observed=("feature_month", "max"),
    first_recorded_failure=("first_failure_month", "min"),
    first_recorded_exit=("first_exit_month", "min"),
    first_ambiguous_exit=("first_ambiguous_exit_month", "min"),
)
first_month = df["feature_month"].min()
last_month = df["feature_month"].max()
lifecycle_summary = pd.Series({
    "Unique securities": len(life),
    "First observed at sample start": life["first_observed"].eq(first_month).sum(),
    "First observed after sample start": life["first_observed"].gt(first_month).sum(),
    "Last observed before sample end": life["last_observed"].lt(last_month).sum(),
    "Last observed at sample end": life["last_observed"].eq(last_month).sum(),
    "Securities with recorded failure date": life["first_recorded_failure"].notna().sum(),
    "Securities with recorded exit date": life["first_recorded_exit"].notna().sum(),
    "Securities with ambiguous exit date": life["first_ambiguous_exit"].notna().sum(),
})
years = pd.Index(range(first_month.year, last_month.year + 1), name="year")
lifecycle_annual = pd.DataFrame(index=years)
for name, column in {
    "first_observed_securities": "first_observed",
    "last_observed_securities": "last_observed",
    "recorded_first_failures": "first_recorded_failure",
    "recorded_first_exits": "first_recorded_exit",
    "recorded_first_ambiguous_exits": "first_ambiguous_exit",
}.items():
    lifecycle_annual[name] = life[column].dt.year.value_counts().reindex(years, fill_value=0)
lifecycle_annual = lifecycle_annual.reset_index()
assert lifecycle_annual["first_observed_securities"].sum() == len(life)
assert lifecycle_annual["last_observed_securities"].sum() == len(life)
display(lifecycle_summary)
display(lifecycle_annual)
del life

In [ ]:
TARGET = "failure_t_plus_1"

SAMPLE_END = pd.Timestamp("2024-12-31")

# Common US security universe
universe_mask = (
    df["global_factor_matched"].eq(1)
    & df["excntry"].eq("USA")
    & df["common"].eq(1)
    & df["primary_sec"].eq(1)
    & df["obs_main"].eq(1)
    & df["exch_main"].eq(1)
).fillna(False)

# Forecast eligibility without requiring a realized outcome
forecast_mask = (
    universe_mask
    & df["at_risk"].eq(1)
    & df["forecast_month"].le(SAMPLE_END)
)

# Supervised estimation and evaluation eligibility
labeled_mask = (
    forecast_mask
    & df[TARGET].isin([0, 1])
)

eligible_panel = df.loc[labeled_mask].copy()

eligible_panel["target"] = (
    eligible_panel[TARGET].astype("int8")
)

print("Forecast-eligible observations:", forecast_mask.sum())
print("Labeled observations:", len(eligible_panel))
print("Positive outcomes:", eligible_panel["target"].sum())

print(
    "Failure prevalence:",
    f"{eligible_panel['target'].mean():.6%}"
)

In [ ]:
# Unique security-month observations
assert not eligible_panel.duplicated(
    ["permno", "feature_month"]
).any()

# Target is strictly binary
assert eligible_panel["target"].isin([0, 1]).all()

# All observations belong to the risk set
assert eligible_panel["at_risk"].eq(1).all()

# Correct forecast horizon
assert eligible_panel["forecast_month"].eq(
    eligible_panel["feature_month"]
    + pd.offsets.MonthEnd(1)
).all()

# Verify alignment against recorded failure dates
assert eligible_panel["target"].eq(
    eligible_panel["first_failure_month"]
    .eq(eligible_panel["forecast_month"])
    .astype("int8")
).all()

# Verify calendar coverage
assert eligible_panel["forecast_month"].le(
    SAMPLE_END
).all()

print("All Part I integrity checks passed.")

# Check all forecast-eligible rows, including unlabeled ones.
assert df.loc[forecast_mask, "at_risk"].eq(1).all()
assert df.loc[forecast_mask, "forecast_month"].eq(
    df.loc[forecast_mask, "feature_month"] + pd.offsets.MonthEnd(1)
).all()
# A flag/date inconsistency warrants review rather than silent relabeling.
risk_timing_diagnostics = pd.Series({
    "At-risk rows with recorded failure on or before feature month": int((
        df["at_risk"].eq(1) & df["first_failure_month"].le(df["feature_month"])
    ).sum()),
    "At-risk rows with recorded exit on or before feature month": int((
        df["at_risk"].eq(1) & df["first_exit_month"].le(df["feature_month"])
    ).sum()),
    "At-risk rows with ambiguous exit on or before feature month": int((
        df["at_risk"].eq(1) & df["first_ambiguous_exit_month"].le(df["feature_month"])
    ).sum()),
})
display(risk_timing_diagnostics)

In [ ]:
positive_events = eligible_panel.loc[
    eligible_panel["target"].eq(1)
]

known_company_events = positive_events.loc[
    positive_events["permco"].notna()
]

sample_summary = pd.Series({
    "Observations": len(eligible_panel),

    "Unique securities":
        eligible_panel["permno"].nunique(),

    "Unique companies":
        eligible_panel["permco"].nunique(),

    "Positive security-month outcomes":
        len(positive_events),

    "Unique PERMCO-month failure events":
        known_company_events[
            ["permco", "forecast_month"]
        ].drop_duplicates().shape[0],

    "Positive outcomes with missing PERMCO":
        positive_events["permco"].isna().sum(),

    "Failure prevalence (%)":
        eligible_panel["target"].mean() * 100,

    "First feature month":
        eligible_panel["feature_month"].min(),

    "Last feature month":
        eligible_panel["feature_month"].max(),

    "First forecast month":
        eligible_panel["forecast_month"].min(),

    "Last forecast month":
        eligible_panel["forecast_month"].max()
})

display(sample_summary)

In [ ]:
# Output directories relative to the notebook location
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
tables_dir = project_root / "outputs" / "tables"

processed_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Preserve prediction-eligible observations,
# including those with unavailable outcomes
export_cols = [
    "permno",
    "permco",
    "gvkey",
    "month_end",
    "feature_month",
    "forecast_month",
    "at_risk",
    "global_factor_matched",
    TARGET
]

panel_export = df.loc[
    forecast_mask, export_cols
].copy()

panel_export = panel_export.rename(
    columns={TARGET: "target"}
)

panel_export["target"] = (
    panel_export["target"].astype("Int8")
)

panel_export["labeled"] = (
    panel_export["target"].notna()
)

# Save the eligible panel
panel_export.to_parquet(
    processed_dir / "part_I_panel.parquet",
    index=False,
    compression="snappy"
)

# Save diagnostic tables
sample_attrition.to_csv(
    tables_dir / "sample_attrition.csv",
    index=False
)

sample_summary.to_csv(
    tables_dir / "sample_summary.csv",
    header=["value"]
)

merge_diagnostics.to_csv(
    tables_dir / "merge_diagnostics.csv",
    header=["value"]
)

censoring_summary.to_csv(
    tables_dir / "censoring_summary.csv",
    index=False
)

continuity_summary.to_csv(
    tables_dir / "panel_continuity.csv",
    header=["value"]
)

print("Part I outputs saved successfully.")

print("\nProcessed panel:")
print(processed_dir / "part_I_panel.parquet")

print("\nDiagnostic tables:")
for path in sorted(tables_dir.glob("*.csv")):
    print(path.name)

# Additional provenance and lifecycle diagnostics.
lifecycle_annual.to_csv(tables_dir / "security_lifecycle_annual.csv", index=False)
lifecycle_summary.to_csv(tables_dir / "security_lifecycle_summary.csv", header=["value"])
risk_timing_diagnostics.to_csv(tables_dir / "risk_timing_diagnostics.csv", header=["value"])